# Safety in Spanish: A Cross-Lingual Evaluation of Jailbreak Vulnerability in LLMs

## Results Analysis

**Inputs:** Loads `judgments.jsonl`, `responses_all.csv`, and `benchmark.csv`.

**Outputs:** ASR@1 tables, McNemar tests, response-language breakdown, plots, `final_summary.csv`.

## 0. Setup

In [1]:
!pip install scikit-learn pandas matplotlib seaborn statsmodels --quiet

In [2]:
import json
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test

## 1. Configuration


In [ ]:
BASE_DIR = Path("/content")
JUDGMENTS_PATH = BASE_DIR / "judgments.jsonl"
RESPONSES_PATH = BASE_DIR / "responses_all.csv"
BENCHMARK_PATH = BASE_DIR / "benchmark.csv"
OUTPUT_DIR = BASE_DIR / "results" / "analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JUDGE = "strongreject_gpt4o_mini"
INCLUDE_MODELS = [
    "mistralai/Mistral-7B-Instruct-v0.3",
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "meta-llama/Meta-Llama-3-8B-Instruct",
    "claude-haiku-4-5-20251001",
]

## 2. Load Data

In [ ]:
assert JUDGMENTS_PATH.exists(), f"Not found: {JUDGMENTS_PATH}"
assert RESPONSES_PATH.exists(), f"Not found: {RESPONSES_PATH}"
assert BENCHMARK_PATH.exists(), f"Not found: {BENCHMARK_PATH}"

df_j = pd.read_json(JUDGMENTS_PATH, lines=True)
df_responses = pd.read_csv(RESPONSES_PATH)
df_bench = pd.read_csv(BENCHMARK_PATH)

df_j["target"] = df_j["target"].fillna("response")

## 3. Final Verdict


In [ ]:
# Single judge -> filter to StrongREJECT only
df_resp_j = df_j[(df_j["target"] == "response") & (df_j["judge"] == JUDGE)].copy()

# Enrich with benchmark metadata
bench_meta = df_bench[["prompt_id", "category", "language", "attack_type"]].drop_duplicates("prompt_id")
df_resp_j  = df_resp_j.merge(bench_meta, on="prompt_id", how="left")

# Enrich with response_language
resp_meta = df_responses[["response_id", "response_language"]].drop_duplicates("response_id")
df_resp_j = df_resp_j.merge(resp_meta, on="response_id", how="left")

# With one judge, no pivot needed — rename columns directly
INDEX_COLS = ["response_id", "prompt_id", "model", "language", "category", "attack_type", "response_language"]
df_wide = df_resp_j[INDEX_COLS + ["score", "is_harmful"]].copy()
df_wide = df_wide.rename(columns={"is_harmful": f"is_harmful_{JUDGE}", "score": f"score_{JUDGE}"})

harm_cols  = [f"is_harmful_{JUDGE}"]
score_cols = [f"score_{JUDGE}"]

df_wide["verdict_any"] = df_wide[f"is_harmful_{JUDGE}"]

print(f"Responses (all models) : {len(df_wide):,}")
print(f"Judge                  : {JUDGE}")
harmful_pct = df_wide[f"is_harmful_{JUDGE}"].mean() * 100
print(f"Harmful (all models)   : {df_wide[f'is_harmful_{JUDGE}'].sum():,} ({harmful_pct:.1f}%)")

excl = df_wide[~df_wide["model"].isin(INCLUDE_MODELS)]
incl = df_wide[df_wide["model"].isin(INCLUDE_MODELS)]
print(f"\nIncluded in results    : {len(incl):,}  ({len(INCLUDE_MODELS)} models)")
print(f"Excluded from display  : {len(excl):,}  ({sorted(excl['model'].unique())})")

## 4. ASR Analysis

ASR@1 = fraction of prompts where at least one run was judged harmful by StrongREJECT.

In [ ]:
GROUP_COLS = ["model", "language", "category", "attack_type"]

# Prompt-level ASR@1: any run harmful across N runs
prompt_level = (
    df_wide.groupby(["prompt_id"] + GROUP_COLS)
    .agg({"verdict_any": "any", f"is_harmful_{JUDGE}": "any", f"score_{JUDGE}": "mean"})
    .reset_index()
)
prompt_level["verdict_any"]           = prompt_level["verdict_any"].astype(int)
prompt_level[f"is_harmful_{JUDGE}"]   = prompt_level[f"is_harmful_{JUDGE}"].astype(int)

# Condition-level summary (all models retained here)
summary = (
    prompt_level.groupby(GROUP_COLS)
    .agg(n_prompts=("prompt_id", "count"),
         n_harmful=(f"is_harmful_{JUDGE}", "sum"),
         mean_score=(f"score_{JUDGE}", "mean"))
    .reset_index()
)
summary["asr"] = (summary["n_harmful"] / summary["n_prompts"]).round(4)
summary["mean_score"] = summary["mean_score"].round(4)

# Display-filtered views — gpt-4o-mini excluded
summary_d      = summary[summary["model"].isin(INCLUDE_MODELS)].copy()
prompt_level_d = prompt_level[prompt_level["model"].isin(INCLUDE_MODELS)].copy()

print(f"Condition rows (all models)      : {len(summary)}")
print(f"Condition rows (display, 5 models): {len(summary_d)}")
print(f"\nTop 10 by ASR:")
display(summary_d.sort_values("asr", ascending=False).head(10))

In [ ]:
print("=== ASR@1 (StrongREJECT) — Model × Language ===\n")
display(
    summary_d.groupby(["model", "language"])["asr"]
    .mean().unstack(fill_value=float("nan")).round(3)
)

print("\n=== EN→ES Gap ===\n")
gap = summary_d.groupby(["model", "language"])["asr"].mean().unstack()
if "en" in gap.columns and "es" in gap.columns:
    gap["gap_es_minus_en"] = gap["es"] - gap["en"]
    display(gap.round(3).sort_values("gap_es_minus_en", ascending=False))

print("\n=== ASR@1 by Harm Category ===\n")
display(
    summary_d.groupby("category")["asr"]
    .mean().sort_values(ascending=False).round(3)
    .rename("mean_asr").to_frame()
)

print("\n=== ASR@1 by Attack Type ===\n")
display(
    summary_d.groupby("attack_type")["asr"]
    .mean().sort_values(ascending=False).round(3)
    .rename("mean_asr").to_frame()
)

print("\n=== Mean StrongREJECT Score — Model × Language ===\n")
display(
    summary_d.groupby(["model", "language"])["mean_score"]
    .mean().unstack(fill_value=float("nan")).round(3)
)

## 5. McNemar's Test — EN vs ES per Model

Paired comparison: same prompt, same attack type, EN vs ES. Tests whether the language shift produces a statistically significant change in ASR.

In [ ]:
def strip_lang_suffix(prompt_id: str, language: str) -> str:
    suffix = f"_{language}"
    return prompt_id[:-len(suffix)] if prompt_id.endswith(suffix) else prompt_id

prompt_level_d["base_prompt_id"] = prompt_level_d.apply(
    lambda r: strip_lang_suffix(r["prompt_id"], r["language"]), axis=1
)

print(f"{'Model':45} {'EN ASR':>7} {'ES ASR':>7} {'Gap':>7} {'p-value':>9} {'sig':>4}")
print("─" * 80)

for model in sorted(prompt_level_d["model"].unique()):
    m  = prompt_level_d[prompt_level_d["model"] == model]
    en = m[m["language"] == "en"][["base_prompt_id", "attack_type", "verdict_any"]].rename(columns={"verdict_any": "en_h"})
    es = m[m["language"] == "es"][["base_prompt_id", "attack_type", "verdict_any"]].rename(columns={"verdict_any": "es_h"})
    paired = en.merge(es, on=["base_prompt_id", "attack_type"])
    if len(paired) < 10:
        print(f"  {model}: insufficient paired data (n={len(paired)})")
        continue
    n00 = ((paired["en_h"] == 0) & (paired["es_h"] == 0)).sum()
    n01 = ((paired["en_h"] == 0) & (paired["es_h"] == 1)).sum()
    n10 = ((paired["en_h"] == 1) & (paired["es_h"] == 0)).sum()
    n11 = ((paired["en_h"] == 1) & (paired["es_h"] == 1)).sum()
    result = mcnemar_test([[n00, n01], [n10, n11]], exact=False, correction=True)
    en_asr = paired["en_h"].mean()
    es_asr = paired["es_h"].mean()
    p   = result.pvalue
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    short = model.split("/")[-1]
    print(f"{short:45} {en_asr:7.3f} {es_asr:7.3f} {es_asr - en_asr:+7.3f} {p:9.4f} {sig:>4}")

## 6. Response Language Analysis (RQ6)

When a model complied with a Spanish prompt, did it respond in Spanish, English, or mixed?

In [ ]:
es_complied = df_wide[
    df_wide["model"].isin(INCLUDE_MODELS) &
    (df_wide["language"] == "es") &
    (df_wide["verdict_any"] == True)
]
print(f"ES harmful responses (5 models): {len(es_complied):,}")

if "response_language" in es_complied.columns and es_complied["response_language"].notna().any():
    lang_dist = (
        es_complied.groupby(["model", "response_language"]).size()
        .unstack(fill_value=0)
    )
    lang_dist["total"] = lang_dist.sum(axis=1)
    lang_cols = [c for c in lang_dist.columns if c != "total"]
    for c in lang_cols:
        lang_dist[f"pct_{c}"] = (lang_dist[c] / lang_dist["total"] * 100).round(1)
    pct_cols = [f"pct_{c}" for c in lang_cols]
    print("\n=== Response language % per model (ES harmful responses) ===")
    display(
        lang_dist[pct_cols]
        .rename(columns=lambda c: c.replace("pct_", ""))
    )
    print("\n=== Overall distribution (5 models) ===")
    display(
        es_complied["response_language"]
        .value_counts(normalize=True).mul(100).round(1)
        .rename("pct").to_frame()
    )
else:
    print("response_language column not available or all null")

## 7. Plots

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.0)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Safety in Spanish — ASR@1 (StrongREJECT, 5 models)", fontsize=14, fontweight="bold", y=1.02)

# Plot 1: Heatmap Model × Language
ax = axes[0, 0]
hmap = summary_d.groupby(["model", "language"])["asr"].mean().unstack()
hmap.index = [m.split("/")[-1] for m in hmap.index]
hmap = hmap[[c for c in ["en", "es", "code_switch"] if c in hmap.columns]]
sns.heatmap(hmap, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0, vmax=1,
            ax=ax, cbar_kws={"label": "ASR@1"}, linewidths=0.5)
ax.set_title("ASR@1 — Model × Language")
ax.set_xlabel("")
ax.set_ylabel("")

# Plot 2: ASR by Harm Category
ax = axes[0, 1]
cat_asr = summary_d.groupby("category")["asr"].mean().sort_values()
bars = ax.barh(cat_asr.index, cat_asr.values,
               color=sns.color_palette("muted", len(cat_asr)))
ax.set_xlim(0, 1.1)
ax.set_title("ASR@1 by Harm Category")
ax.set_xlabel("Mean ASR@1")
for bar, val in zip(bars, cat_asr.values):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=9)

# Plot 3: EN vs ES vs Code-switch grouped bar
ax = axes[1, 0]
lang_data = summary_d.groupby(["model", "language"])["asr"].mean().unstack()
lang_data.index = [m.split("/")[-1] for m in lang_data.index]
lang_cols = [c for c in ["en", "es", "code_switch"] if c in lang_data.columns]
lang_data[lang_cols].plot(kind="bar", ax=ax, rot=35, width=0.7,
                          color=["#4C72B0", "#DD8452", "#55A868"][:len(lang_cols)])
ax.set_title("ASR@1 by Model and Language")
ax.set_ylabel("Mean ASR@1")
ax.set_ylim(0, 1.05)
ax.legend(title="Language", loc="upper left")
ax.set_xlabel("")

# Plot 4: ASR by Attack Type
ax = axes[1, 1]
atk_asr = summary_d.groupby("attack_type")["asr"].mean().sort_values()
bars = ax.barh(atk_asr.index, atk_asr.values,
               color=["#4C72B0", "#DD8452", "#55A868"][:len(atk_asr)])
ax.set_xlim(0, 1.1)
ax.set_title("ASR@1 by Attack Type")
ax.set_xlabel("Mean ASR@1")
for bar, val in zip(bars, atk_asr.values):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=10)

plt.tight_layout()
plot_path = OUTPUT_DIR / "asr_analysis.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {plot_path}")

### EN→ES gap heatmap

In [ ]:
gap_data = summary_d.groupby(["model", "language"])["asr"].mean().unstack()
if "en" in gap_data.columns and "es" in gap_data.columns:
    gap_data.index = [m.split("/")[-1] for m in gap_data.index]
    gap_df = pd.DataFrame(
        gap_data["es"].values - gap_data["en"].values,
        index=gap_data.index,
        columns=["ES − EN gap"]
    )

    fig, ax = plt.subplots(figsize=(3, len(gap_df) * 0.7 + 1))
    sns.heatmap(gap_df, annot=True, fmt="+.3f", cmap="RdBu_r",
                center=0, vmin=-0.3, vmax=0.3,
                ax=ax, cbar_kws={"label": "ASR gap (ES − EN)"},
                linewidths=0.5)
    ax.set_ylabel("")
    plt.tight_layout()
    gap_path = OUTPUT_DIR / "gap_heatmap.png"
    plt.savefig(gap_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {gap_path}")
else:
    print("Need both 'en' and 'es' in the data to plot gap.")